## IMPORTATION DES BIBLIOTHEQUES ET PARAMETRES GLOBAUX

In [2]:
# ---  Préambule : imports ---
%matplotlib qt
import os, sys, importlib
from pathlib import Path
import ipynbname

# ---  Définition des chemins de base ---
base_path = Path(ipynbname.path()).parent.parent.parent  # racine du projet
path_data = base_path / 'DATA'

# ---  Ajouter dossier scripts au Python path ---
scripts_path = base_path / 'Code' / 'scripts'
sys.path.append(str(scripts_path.resolve()))

# ---  Import des modules annexes ---
import fonctions_annexes_biodiv
importlib.reload(fonctions_annexes_biodiv)
from fonctions_annexes_biodiv import generer_dictionnaire_taxonomie

from fusion_data import *  # importe toutes les fonctions utilitaires d'IO
from formatage_data import *


In [ ]:
# ---  Fichiers du Gabon ---
country_file = path_data / 'SIG_global' / 'gabon.shp'
gbif_file = path_data / 'GBIF' / 'raw' / 'observations_gabon.csv'

In [ ]:
from scripts.geo_utils import *
from scripts.gbif_utils import *
from scripts.grid_fusion import *
from scripts.io_utils import *

# --- Paramètres ---
country_file = '../SIG_global/france.shp'
gbif_file = '../DATA/GBIF/raw/observations.csv'

# --- Charger les données ---
country_gdf = load_geospatial_data(country_file)
df_gbif = load_csv_file(gbif_file)

# --- Créer la grille ---
grid_gdf = create_country_grid(country_gdf, cell_size_km=10)

# --- Associer les observations à la grille ---
df_gbif_geo = add_grid_to_country(df_gbif, grid_gdf)

# --- Traiter et normaliser ---
df_gbif_proc = process_biodiv_data(df_gbif_geo)

# --- Fusion éventuelle des cellules ---
df_gbif_fused = fusion_cells_by_obs(df_gbif_proc, min_obs=10)

# --- Sauvegarder résultat ---
save_csv_file(df_gbif_fused, '../DATA/GBIF/processed/gbif_processed.csv')


### Fusion

## INAT

In [ ]:
# ==========================
# Notebook : Biodiversité Gabon
# ==========================

# --- 0️⃣ Préambule : imports ---
%matplotlib qt
import os, sys, importlib
from pathlib import Path
import ipynbname

# --- 1️⃣ Définition des chemins de base ---
base_path = Path(ipynbname.path()).parent.parent.parent  # racine du projet
path_data = base_path / 'DATA'

# --- 2️⃣ Fichiers du Gabon ---
country_file = path_data / 'SIG_global' / 'gabon.shp'
gbif_file = path_data / 'GBIF' / 'raw' / 'observations_gabon.csv'

# --- 3️⃣ Ajouter dossier scripts au Python path ---
scripts_path = base_path / 'Code' / 'scripts'
sys.path.append(str(scripts_path.resolve()))

# --- 4️⃣ Import des modules annexes ---
import fonctions_annexes_biodiv
importlib.reload(fonctions_annexes_biodiv)
from fonctions_annexes_biodiv import generer_dictionnaire_taxonomie

from io_utils import *  # importe toutes les fonctions utilitaires d'IO

# ==========================
# --- 5️⃣ Lecture des données ---
# ==========================
# Lecture shapefile du Gabon
from geopandas import read_file
gabon_shape = read_file(country_file)

# Lecture observations GBIF
df_gbif = load_csv_file(gbif_file)  # fonction importée depuis io_utils

# ==========================
# --- 6️⃣ Nettoyage / Préparation ---
# ==========================
# Exemple : normaliser noms scientifiques et filtrer les observations
df_gbif['nomScientifique'] = df_gbif['nomScientifique'].str.strip().str.capitalize()

# Générer dictionnaire taxonomie
taxo_dict = generer_dictionnaire_taxonomie(df_gbif)

# ==========================
# --- 7️⃣ Analyse exploratoire ---
# ==========================
import matplotlib.pyplot as plt

# Histogramme des observations par espèce
obs_par_espece = df_gbif.groupby('nomScientifique')['nombreObs'].sum().sort_values(ascending=False)
plt.figure(figsize=(12,6))
obs_par_espece.plot(kind='bar')
plt.title("Nombre d'observations par espèce - Gabon")
plt.ylabel("Nombre d'observations")
plt.xlabel("Espèce")
plt.tight_layout()
plt.show()

# ==========================
# --- 8️⃣ Export / sauvegarde ---
# ==========================
output_file = path_data / 'GBIF/processed' / 'gbif_gabon_taxo.csv'
save_csv_file(df_gbif, output_file)  # fonction importée depuis io_utils
